In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\Sandeep\anaconda3\envs\textS\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\Sandeep\anaconda3\envs\textS\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.s

In [3]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))
print("PyTorch version:", torch.__version__)

CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 3050 Laptop GPU
PyTorch version: 2.2.2+cu121


In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
import torch
print("Torch version:", torch.__version__)
print("Torch location:", torch.__file__)

import transformers
print("Transformers version:", transformers.__version__)

Torch version: 2.2.2+cu121
Torch location: c:\Users\Sandeep\anaconda3\envs\textS\lib\site-packages\torch\__init__.py
Transformers version: 4.40.0


### Dataset load

In [6]:
dataset = load_dataset("csv", data_files={
    "train": r"C:\Users\Sandeep\Desktop\Projects\datasets\samsum-train.csv",
    "test": r"C:\Users\Sandeep\Desktop\Projects\datasets\samsum-test.csv",
    "validation": r"C:\Users\Sandeep\Desktop\Projects\datasets\samsum-validation.csv"
}, keep_in_memory=False)

Generating train split: 0 examples [00:00, ? examples/s]

: 

In [9]:
# Number of rows in each split
print("Train rows:", dataset["train"].num_rows)
print("Test rows:", dataset["test"].num_rows)
print("Validation rows:", dataset["validation"].num_rows)

# Column names
print("Columns:", dataset["train"].column_names)

Train rows: 14732
Test rows: 819
Validation rows: 818
Columns: ['id', 'dialogue', 'summary']


### Load tokeniser

In [12]:
model_checkpoint = "t5-small"   # we'll start with this, upgrade later
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [13]:
tokenizer

T5Tokenizer(name_or_path='t5-small', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=Tru

In [14]:
sample = dataset["train"][0]

inputs = tokenizer(sample["dialogue"], truncation=True, max_length=512)
target = tokenizer(sample["summary"], truncation=True, max_length=128)

print("Input tokens:", len(inputs["input_ids"]))
print("Target tokens:", len(target["input_ids"]))
print("Sample dialogue:", sample["dialogue"][:200])
print("Sample summary:", sample["summary"])

Input tokens: 28
Target tokens: 11
Sample dialogue: Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)
Sample summary: Amanda baked cookies and will bring Jerry some tomorrow.


In [ ]:
# Filter out None values before tokenising
dialogues = [x for x in dataset["train"]["dialogue"] if isinstance(x, str)]
summaries = [x for x in dataset["train"]["summary"] if isinstance(x, str)]

# For T5, use with_prefix approach or just decode directly
dialogue_lengths = [len(tokenizer.encode(x)) for x in dialogues]
summary_lengths  = [len(tokenizer.encode(x)) for x in summaries]

print("Dialogue — max:", max(dialogue_lengths), "| avg:", int(np.mean(dialogue_lengths)))
print("Summary  — max:", max(summary_lengths),  "| avg:", int(np.mean(summary_lengths)))

Dialogue — max: 1153 | avg: 148
Summary  — max: 94 | avg: 28


In [29]:
# Find the null values
total = len(dataset["train"]["dialogue"])
null_d = sum(1 for x in dataset["train"]["dialogue"] if x is None)
null_s = sum(1 for x in dataset["train"]["summary"] if x is None)
empty_d = sum(1 for x in dataset["train"]["dialogue"] if x == "")
empty_s = sum(1 for x in dataset["train"]["summary"] if x == "")

print(f"Total rows:        {total}")
print(f"Null dialogues:    {null_d} ({null_d/total*100:.1f}%)")
print(f"Null summaries:    {null_s} ({null_s/total*100:.1f}%)")
print(f"Empty dialogues:   {empty_d}")
print(f"Empty summaries:   {empty_s}")

# Peek at a few null rows to understand why
null_indices = [i for i, x in enumerate(dataset["train"]["dialogue"]) if x is None]
print("\nSample null rows:")
for i in null_indices[:5]:
    print(dataset["train"][i])

Total rows:        14732
Null dialogues:    1 (0.0%)
Null summaries:    0 (0.0%)
Empty dialogues:   0
Empty summaries:   0

Sample null rows:
{'id': '13828807', 'dialogue': None, 'summary': 'problem with visualization of the content'}


In [30]:
# Remove the null values
dataset = dataset.filter(lambda x: x["dialogue"] is not None and x["summary"] is not None)

# Confirm
print("Train rows after cleaning:", dataset["train"].num_rows)

Filter: 100%|██████████| 818/818 [00:00<00:00, 28121.08 examples/s]

Train rows after cleaning: 14731


In [31]:
# TOkenizing the complete dataset
max_input_length = 512
max_target_length = 128
prefix = "summarize: "

def preprocess(examples):
    inputs = [prefix + doc for doc in examples["dialogue"]]
    
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )
    
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply to all splits at once
tokenized_dataset = dataset.map(preprocess, batched=True)
print(tokenized_dataset)

Map: 100%|██████████| 818/818 [00:00<00:00, 4207.01 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
})


### Load the model

In [40]:
# Load the model
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
model = model.to(device)
print("Model loaded!", model)

ImportError: 
AutoModelForSeq2SeqLM requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.
